# 🚀 AI 3D Studio v6.0 — Google Colab Production Deployment
### State-of-the-Art Generative 3D Mesh Engine (ComfyUI + ComfyUI-3D-Pack)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Silentzx2/AI_Studio/blob/main/colab.ipynb)


In [ ]:
# @title 🔍 Step 1: Hardware & GPU Verification
# @markdown Run this cell to verify GPU hardware, VRAM, system memory, and CUDA driver.

import os
import sys
import shutil
import subprocess

print("═" * 68)
print("  🚀 AI 3D STUDIO — HARDWARE & RUNTIME DIAGNOSTIC")
print("═" * 68)

# 1. Check GPU presence via nvidia-smi
has_gpu = False
gpu_name = "None"
gpu_vram = 0.0
driver_ver = "None"

try:
    smi_raw = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader,nounits"],
        stderr=subprocess.STDOUT,
        text=True
    ).strip()
    if smi_raw:
        lines = smi_raw.splitlines()
        for idx, line in enumerate(lines):
            parts = [p.strip() for p in line.split(",")]
            g_name = parts[0]
            g_vram = float(parts[1]) if len(parts) > 1 else 0
            g_drv = parts[2] if len(parts) > 2 else "Unknown"
            print(f"  [GPU {idx}]  : {g_name} | VRAM: {g_vram:.0f} MB ({g_vram/1024:.1f} GB) | Driver: {g_drv}")
            if idx == 0:
                has_gpu = True
                gpu_name = g_name
                gpu_vram = g_vram
                driver_ver = g_drv
except Exception:
    pass

# 2. Check System RAM & Disk
try:
    import psutil
    ram_gb = psutil.virtual_memory().total / (1024**3)
    ram_avail_gb = psutil.virtual_memory().available / (1024**3)
    cpu_cores = psutil.cpu_count(logical=True)
except ImportError:
    ram_gb = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / (1024**3)
    ram_avail_gb = ram_gb
    cpu_cores = os.cpu_count() or 2

disk_free_gb = shutil.disk_usage("/").free / (1024**3)
disk_total_gb = shutil.disk_usage("/").total / (1024**3)

print(f"  [CPU]    : {cpu_cores} vCPU threads")
print(f"  [RAM]    : {ram_gb:.1f} GB Total ({ram_avail_gb:.1f} GB Available)")
print(f"  [DISK]   : {disk_free_gb:.1f} GB Free / {disk_total_gb:.1f} GB Total")
print("═" * 68)

if has_gpu:
    print("  ✅ COMPATIBLE GPU DETECTED!")
    if gpu_vram >= 14000:
        print(f"  💡 {gpu_name} ({gpu_vram/1024:.1f} GB VRAM) is fully supported.")
        print("     Supported: TripoSG, TRELLIS, Hunyuan3D-2mini, Hunyuan3D-2.1.")
    else:
        print(f"  ⚠️ GPU has {gpu_vram/1024:.1f} GB VRAM. TripoSG and TRELLIS will run well;")
        print("     High-VRAM models (Hunyuan3D-2.1) may require lower batch/mesh resolution.")
else:
    print("  ❌ NO GPU DETECTED!")
    print("  👉 Please enable GPU acceleration:")
    print("     1. Click 'Runtime' in the Colab top navigation bar.")
    print("     2. Select 'Change runtime type'.")
    print("     3. Set Hardware accelerator to 'T4 GPU' or 'A100 GPU'.")
    print("     4. Click 'Save' and re-run this cell.")
print("═" * 68)


In [ ]:
# @title 📥 Step 2: Git Clone / Sync Repository
# @markdown Clones the AI 3D Studio repository if missing, or pulls latest updates if already present.

import os
import sys
import subprocess
from pathlib import Path

# Clone if directory doesn't exist: git clone https://github.com/Silentzx2/AI_Studio.git /content/AI_Studio
# Sync if directory exists: git pull
REPO_URL = "https://github.com/Silentzx2/AI_Studio.git"
TARGET_DIR = Path("/content/AI_Studio")

# If running in local dev / non-Colab, preserve current repository if scripts/colab.sh exists
if not TARGET_DIR.exists() and Path("scripts/colab.sh").exists():
    TARGET_DIR = Path(".").resolve()

print(f"📂 Target Workspace: {TARGET_DIR}")

if TARGET_DIR.exists() and (TARGET_DIR / ".git").exists():
    print("🔄 Existing repository detected. Pulling latest updates from origin...")
    try:
        res = subprocess.run(["git", "pull", "--rebase"], cwd=str(TARGET_DIR), capture_output=True, text=True)
        if res.returncode == 0:
            print(res.stdout.strip() or "  Already up to date.")
        else:
            print("⚠️ Rebase pull notice, attempting fast-forward pull...")
            subprocess.run(["git", "pull"], cwd=str(TARGET_DIR), check=False)
    except Exception as e:
        print(f"⚠️ Git pull warning: {e}")
else:
    if TARGET_DIR.exists():
        import shutil
        shutil.rmtree(str(TARGET_DIR), ignore_errors=True)
    print(f"🚀 Cloning repository from {REPO_URL}...")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(TARGET_DIR)], check=True)

# Change directory
os.chdir(str(TARGET_DIR))
%cd {TARGET_DIR}

# Sanity check critical paths
assert Path("scripts/colab.sh").exists(), "Error: scripts/colab.sh not found!"
assert Path("scripts/colab_watch.sh").exists(), "Error: scripts/colab_watch.sh not found!"
assert Path("backend").exists(), "Error: backend directory not found!"

# Display active commit
try:
    commit = subprocess.check_output(["git", "log", "-1", "--oneline"], cwd=str(TARGET_DIR), text=True).strip()
    branch = subprocess.check_output(["git", "branch", "--show-current"], cwd=str(TARGET_DIR), text=True).strip()
    print(f"
✅ Repository ready on branch '{branch}': {commit}")
except Exception:
    print(f"
✅ Active directory: {os.getcwd()}")


In [ ]:
# @title 🚀 Step 3: Complete One-Click Production Launcher
# @markdown Runs full bootstrap (swap, PostgreSQL, Redis, venvs, PyTorch CUDA, Next.js build, Cloudflare tunnels, and supervisor).

import os
import re
import sys
import time
import subprocess
from pathlib import Path
from IPython.display import HTML, Javascript, display

# Ensure working directory is AI_Studio root
if Path("/content/AI_Studio").exists():
    os.chdir("/content/AI_Studio")

# Configuration for Colab
os.environ["DISABLE_DISK_CHECK"] = "1"
os.environ["BACKEND_PYTHON_VERSION"] = "3.12"

# Activate browser-level keep-alive to prevent Colab idle timeouts
keepalive_path = Path("scripts/colab_keepalive_js.py")
if keepalive_path.exists():
    try:
        exec(keepalive_path.read_text())
    except Exception as exc:
        print(f"[Keepalive] Note: {exc}")

print("
" + "═" * 68)
print("  🚀 LAUNCHING AI 3D STUDIO ONE-CLICK DEPLOYMENT")
print("  ⚙️ Provisioning Swap, CUDA 12.4, PostgreSQL, Redis, Backend,")
print("     ComfyUI Engine + ComfyUI-3D-Pack, Next.js Frontend & Cloudflare Tunnels...")
print("═" * 68 + "
")

cmd = ["bash", "scripts/colab.sh", "--setup"]
process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    universal_newlines=True
)

cf_frontend_url = None
cf_api_url = None
tunnel_card_rendered = False

def render_tunnel_card(f_url, a_url):
    card_html = f"""
    <div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
                background: linear-gradient(135deg, #181825 0%, #1e1e2e 100%);
                border: 2px solid #89b4fa; border-radius: 14px; padding: 22px 26px; margin: 18px 0;
                box-shadow: 0 10px 30px rgba(0, 0, 0, 0.45); color: #cdd6f4;">
      <div style="display: flex; align-items: center; justify-content: space-between; margin-bottom: 14px;">
        <h2 style="margin: 0; color: #89b4fa; font-size: 22px; font-weight: 700; display: flex; align-items: center; gap: 10px;">
          ✨ AI 3D Studio is Ready!
        </h2>
        <span style="background: #a6e3a1; color: #11111b; font-weight: 700; font-size: 12px; padding: 4px 12px; border-radius: 20px;">
          ONLINE
        </span>
      </div>
      <p style="margin: 0 0 16px 0; color: #bac2de; font-size: 14px; line-height: 1.4;">
        Open the 3D Generation Web Interface in your browser by clicking the button below:
      </p>
      <div style="display: flex; flex-wrap: wrap; gap: 14px; margin-bottom: 18px;">
        <a href="{f_url}" target="_blank"
           style="background: linear-gradient(135deg, #89b4fa 0%, #74c7ec 100%); color: #11111b;
                  font-weight: 700; text-decoration: none; padding: 12px 24px; border-radius: 8px;
                  font-size: 15px; box-shadow: 0 4px 14px rgba(137, 180, 250, 0.4); display: inline-block;">
          🌐 Open Web Application
        </a>
        <a href="{a_url}/docs" target="_blank"
           style="background: #313244; color: #cdd6f4; border: 1px solid #585b70;
                  font-weight: 600; text-decoration: none; padding: 12px 20px; border-radius: 8px;
                  font-size: 15px; display: inline-block;">
          📚 Swagger API Docs
        </a>
      </div>
      <div style="background: rgba(0,0,0,0.3); border-radius: 8px; padding: 12px 16px; font-size: 13px; color: #a6adc8; line-height: 1.6;">
        <div><strong>Frontend URL:</strong> <a href="{f_url}" target="_blank" style="color: #89dceb; word-break: break-all;">{f_url}</a></div>
        <div><strong>Backend API URL:</strong> <a href="{a_url}" target="_blank" style="color: #89dceb; word-break: break-all;">{a_url}</a></div>
        <div style="margin-top: 6px; font-size: 12px; color: #6c7086;">
          🔒 Tunnels provided via Cloudflare. Supervisor runs continuously in foreground.
        </div>
      </div>
    </div>
    """
    display(HTML(card_html))

try:
    for line in iter(process.stdout.readline, ''):
        print(line, end='', flush=True)

        # Check for tunnel URLs in output or in file
        if not tunnel_card_rendered:
            f_file = Path(".cloudflare_tunnels/3000.url")
            a_file = Path(".cloudflare_tunnels/8000.url")
            if f_file.exists() and a_file.exists():
                f_text = f_file.read_text().strip()
                a_text = a_file.read_text().strip()
                if f_text and a_text:
                    cf_frontend_url = f_text
                    cf_api_url = a_text
                    render_tunnel_card(cf_frontend_url, cf_api_url)
                    tunnel_card_rendered = True
    process.wait()
except KeyboardInterrupt:
    print("\n🛑 Interrupted by user. Monitored background services may still be running.")
    process.terminate()


In [ ]:
# @title 🛠️ Step 4: Service Manager & Maintenance Controls
# @markdown Control services: Check Health, Tail Logs, Restart Services, Stop Services, or Refresh Cloudflare Tunnels.

import os
import sys
import time
import subprocess
import urllib.request
from pathlib import Path
from IPython.display import HTML, display

PROJECT_ROOT = Path("/content/AI_Studio" if Path("/content/AI_Studio").exists() else ".").resolve()

def check_status():
    print("═" * 65)
    print("  📊 AI 3D STUDIO — SERVICE HEALTH & STATUS")
    print("═" * 65)

    # 1. API Health
    api_ok = False
    try:
        req = urllib.request.Request("http://127.0.0.1:8000/api/v1/health", headers={"User-Agent": "ColabCheck"})
        with urllib.request.urlopen(req, timeout=3) as resp:
            if resp.status == 200:
                api_ok = True
    except Exception:
        pass
    print(f"  {'🟢' if api_ok else '🔴'} FastAPI Backend  : {'Healthy (port 8000)' if api_ok else 'Not Responding'}")

    # 2. Frontend Health
    frontend_ok = False
    try:
        req = urllib.request.Request("http://127.0.0.1:3000/", headers={"User-Agent": "ColabCheck"})
        with urllib.request.urlopen(req, timeout=3) as resp:
            if resp.status == 200:
                frontend_ok = True
    except Exception:
        pass
    print(f"  {'🟢' if frontend_ok else '🔴'} Next.js Frontend: {'Healthy (port 3000)' if frontend_ok else 'Not Responding'}")

    # 3. ComfyUI Engine
    comfy_ok = False
    try:
        req = urllib.request.Request("http://127.0.0.1:8188/system_stats", headers={"User-Agent": "ColabCheck"})
        with urllib.request.urlopen(req, timeout=3) as resp:
            if resp.status == 200:
                comfy_ok = True
    except Exception:
        pass
    print(f"  {'🟢' if comfy_ok else '🔴'} ComfyUI Engine  : {'Healthy (port 8188)' if comfy_ok else 'Not Responding'}")

    # 4. Supervisor / Watchdog
    sup_pid_file = PROJECT_ROOT / ".pids" / "supervisor.pid"
    sup_alive = False
    if sup_pid_file.exists():
        spid = sup_pid_file.read_text().strip()
        if spid.isdigit():
            try:
                os.kill(int(spid), 0)
                sup_alive = True
            except OSError:
                pass
    print(f"  {'🟢' if sup_alive else '🟡'} Watchdog         : {'Active (auto-restart enabled)' if sup_alive else 'Inactive'}")

    # 5. Cloudflare Tunnels
    f_url_file = PROJECT_ROOT / ".cloudflare_tunnels" / "3000.url"
    a_url_file = PROJECT_ROOT / ".cloudflare_tunnels" / "8000.url"
    f_url = f_url_file.read_text().strip() if f_url_file.exists() else None
    a_url = a_url_file.read_text().strip() if a_url_file.exists() else None

    print("═" * 65)
    if f_url and a_url:
        print(f"  🌐 Public Web UI : {f_url}")
        print(f"  📚 Public API    : {a_url}")
        display(HTML(f'''
        <div style="margin: 10px 0;">
          <a href="{f_url}" target="_blank" style="background:#89b4fa; color:#11111b; padding:8px 16px; border-radius:6px; font-weight:bold; text-decoration:none; margin-right:10px;">🌐 Open Frontend</a>
          <a href="{a_url}/docs" target="_blank" style="background:#313244; color:#cdd6f4; border:1px solid #585b70; padding:8px 16px; border-radius:6px; font-weight:bold; text-decoration:none;">📚 API Docs</a>
        </div>
        '''))
    else:
        print("  ⚠️ Cloudflare tunnel URLs not found. Run refresh_tunnels() to create them.")
    print("═" * 65)

def view_logs(service="api", lines=40):
    log_map = {
        "api": PROJECT_ROOT / "logs" / "api.log",
        "comfyui": PROJECT_ROOT / "logs" / "comfyui.log",
        "frontend": PROJECT_ROOT / "logs" / "frontend.log",
        "watchdog": PROJECT_ROOT / "logs" / "watchdog.log",
        "build": PROJECT_ROOT / "logs" / "frontend_build.log"
    }
    file_path = log_map.get(service, PROJECT_ROOT / "logs" / f"{service}.log")
    print(f"📄 Tail of {file_path.name} (last {lines} lines):\\n" + "─" * 60)
    if file_path.exists():
        text = file_path.read_text()
        tail = text.splitlines()[-lines:]
        print("\\n".join(tail) if tail else "(Log file is empty)")
    else:
        print(f"Log file not found: {file_path}")
    print("─" * 60)

def restart_services():
    print("🔄 Restarting all AI 3D Studio services...")
    subprocess.run(["bash", "scripts/colab.sh", "--restart"], cwd=str(PROJECT_ROOT))
    time.sleep(3)
    check_status()

def stop_services():
    print("🛑 Stopping all AI 3D Studio services...")
    subprocess.run(["bash", "scripts/colab.sh", "--stop"], cwd=str(PROJECT_ROOT))
    time.sleep(2)
    check_status()

def refresh_tunnels():
    print("🌐 Regenerating Cloudflare Tunnels for ports 3000 & 8000...")
    cf_dir = PROJECT_ROOT / ".cloudflare_tunnels"
    cf_dir.mkdir(exist_ok=True)
    subprocess.run(["pkill", "-f", "cloudflared tunnel"], check=False)
    time.sleep(1)
    
    for port, name in [(3000, "Frontend"), (8000, "API")]:
        log_file = cf_dir / f"{port}.log"
        pid_file = cf_dir / f"{port}.pid"
        url_file = cf_dir / f"{port}.url"
        with open(log_file, "w") as out:
            p = subprocess.Popen(
                ["cloudflared", "tunnel", "--url", f"http://localhost:{port}", "--no-autoupdate"],
                stdout=out, stderr=out
            )
            pid_file.write_text(str(p.pid))
        
        # Wait up to 10s for trycloudflare URL
        time.sleep(4)
        if log_file.exists():
            import re
            m = re.search(r"https://[-a-zA-Z0-9]+\.trycloudflare\.com", log_file.read_text())
            if m:
                url = m.group(0)
                url_file.write_text(url)
                print(f"  ✓ {name} (port {port}) -> {url}")
            else:
                print(f"  ⚠️ {name} tunnel launched; check logs/{port}.log for URL")
    
    check_status()

# Default action: display status
check_status()

print("\\n💡 Maintenance commands available in this notebook session:")
print("  • check_status()       -> Show health check and tunnel links")
print("  • view_logs('api')     -> View FastAPI logs (or 'comfyui', 'frontend', 'watchdog')")
print("  • restart_services()   -> Clean restart of all services")
print("  • stop_services()      -> Stop all services and release ports")
print("  • refresh_tunnels()    -> Re-generate fresh public Cloudflare tunnel URLs")
